<a href="https://colab.research.google.com/github/HoussemMouradi/OpenRadioss2PhysicsNeMo/blob/main/OpenRadioss2PhysicsNeMo_linux.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenRadioss2PhysicsNeMo Dataset Generation

This notebook automates sequential OpenRadioss simulations to generate training datasets for PhysicsNeMo machine learning models. By following these steps, you will install all prerequisites needed to create your own dataset and run the PhysicsNeMo example.

Specifically, this repository provides a full pipeline for generating crash simulation data using OpenRadioss. The generated datasets are formatted to train AI models within the [PhysicsNeMo framework](https://github.com/NVIDIA/physicsnemo/tree/main/examples/structural_mechanics/crash), using the [OpenRadioss Bumper Beam example](http://openradioss.atlassian.net/wiki/spaces/OPENRADIOSS/pages/11075585/Bumper+Beam) as a baseline. Raw data is generated by iteratively modifying the shell thickness and pole position across successive simulation runs.

The OpenRadioss ANIM output files are then converted to d3plot format using the [Vortex-Radioss](https://github.com/Vortex-CAE/Vortex-Radioss) tool, enabling compatibility with the d3plot reader available in the PhysicsNeMo crash example.

**To get started**, download OpenRadioss from the [official releases page](https://github.com/OpenRadioss/OpenRadioss/releases). If you only need the pre-generated dataset for training, it is available on [Hugging Face](https://huggingface.co/datasets/AIRBORNEPANDA/BumperBeamCrashExample).


## Step 1 — Download and Set Up OpenRadioss

Download the pre-built OpenRadioss Linux binaries from the official GitHub releases, extract them into the Colab environment, and set the executable permissions on the Starter and Engine binaries.


In [ ]:
# Download the latest OpenRadioss Linux binaries from the official GitHub releases page.
!wget https://github.com/OpenRadioss/OpenRadioss/releases/download/latest-20251211/OpenRadioss_linux64.zip -O OpenRadioss_linux64.zip

# Extract the archive into /content/. This creates an OpenRadioss/ folder
# containing the Starter, Engine, and all required runtime libraries.
!unzip -q OpenRadioss_linux64.zip -d /content/

# Grant execute permission to the Starter and Engine binaries.
!chmod +x /content/OpenRadioss/exec/*

print("OpenRadioss downloaded and extracted successfully!")


--2026-04-27 20:13:26--  https://github.com/OpenRadioss/OpenRadioss/releases/download/latest-20251211/OpenRadioss_linux64.zip
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/490751204/deab1bb1-e96f-4dcb-bd44-35a4222eff11?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-27T20%3A48%3A13Z&rscd=attachment%3B+filename%3DOpenRadioss_linux64.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-27T19%3A48%3A12Z&ske=2026-04-27T20%3A48%3A13Z&sks=b&skv=2018-11-09&sig=VoOYFxRxPBCeTw13zQqCz%2FoKFYrr9KVmUjFzRR3ICDQ%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NzMyMjYwNiwibmJmIjoxNzc3MzIwODA2LCJwYXRoIjoicmV

## Step 2 — Install Vortex-Radioss Dependencies

Install the Python dependencies and clone the Vortex-Radioss package required to convert OpenRadioss ANIM output files into the d3plot format expected by PhysicsNeMo.

- **lasso-python**: provides the d3plot read/write interface used internally by Vortex-Radioss.
- **tqdm**: displays progress bars during batch conversion.

A sparse checkout is used to clone only the `vortex_radioss` sub-folder of the repository, avoiding downloading the full repository history and keeping the Colab environment lean.


In [ ]:
# lasso-python: provides the d3plot read/write interface used internally by Vortex-Radioss
!pip install lasso-python

# tqdm: progress bar library used during batch file conversion
!pip install tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 102.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
^C


In [ ]:
import sys

# Use a sparse checkout to clone only the vortex_radioss sub-folder,
# avoiding downloading the full repository history and keeping the environment lean.
!rm -r /vortex_radioss
!rm -rf .git vortex_radioss
!git init
!git remote add -f origin https://github.com/Vortex-CAE/Vortex-Radioss
!git config core.sparseCheckout true
!echo "vortex_radioss" >> .git/info/sparse-checkout
!git pull origin main

# Try the standard import path first; fall back to an alternative path if it fails
try:
    from vortex_radioss.animtod3plot.Anim_to_D3plot import readAndConvert
    print("Vortex-Radioss imported successfully!")
except ImportError as e:
    print(f"Import error: {e}")
    print("Attempting alternative import path...")
    sys.path.insert(0, '/content/Vortex-Radioss/vortex_radioss')
    from animtod3plot.Anim_to_D3plot import readAndConvert
    print("Vortex-Radioss imported successfully via alternative path!")


rm: cannot remove '/vortex_radioss': No such file or directory
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
Updating origin
remote: Enumerating objects: 651, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 651 (delta 17), reused 12 (delta 12), pack-reused 623 (from 2)
Receiving objects: 100% (651/651), 27.57 MiB | 18.52 MiB/s, done.
Resolving deltas: 100% (300/300), done.
From https://github.com/Vortex-CAE/Vortex-Radioss
 * [new branch]  

## Step 3 — Create the OpenRadioss Run Script

Generate a Linux shell script (`runradioss.sh`) that configures the required environment variables and launches the OpenRadioss Starter and Engine executables sequentially for a given simulation.

The script accepts four positional arguments:

| Argument | Description |
|---|---|
| `$1` | Path to the OpenRadioss installation directory |
| `$2` | Path to the simulation folder |
| `$3` | Simulation file name (without the `_0000`/`_0001` suffix) |
| `$4` | Number of CPU threads to use |

> **Reference:** Environment variable setup follows the official [OpenRadioss installation guide](https://github.com/OpenRadioss/OpenRadioss/blob/main/INSTALL.md).


In [ ]:
# Define the content of the Linux shell script.
# The script accepts 4 positional arguments:
#   $1 = OpenRadioss installation path
#   $2 = Simulation folder path
#   $3 = Simulation file name (without the _0000/_0001 suffix)
#   $4 = Number of CPU threads
#
# It sets the required environment variables (runtime libraries, config paths, stack size),
# then runs the Starter (pre-processor) followed by the Engine (solver) sequentially.
runradioss_sh_content = '''#!/bin/bash
# OpenRadioss Linux Run Script

OPENRADIOSS_PATH=$1
sim_path=$2
sim_file_name=$3
number_of_threads=$4

export RAD_CFG_PATH=${OPENRADIOSS_PATH}/hm_cfg_files
export RAD_H3D_PATH=${OPENRADIOSS_PATH}/extlib/h3d/lib/linux64
export LD_LIBRARY_PATH=${OPENRADIOSS_PATH}/extlib/hm_reader/linux64:${LD_LIBRARY_PATH}
export OMP_STACKSIZE=400m

cd ${sim_path}

# Run starter (pre-processor): reads the _0000.rad input deck and prepares the model
${OPENRADIOSS_PATH}/exec/starter_linux64_gf -i ${sim_path}/${sim_file_name}_0000.rad -nt ${number_of_threads} -np 1

# Run engine (solver): reads the _0001.rad control deck and runs the time-marching simulation
${OPENRADIOSS_PATH}/exec/engine_linux64_gf_sp -i ${sim_path}/${sim_file_name}_0001.rad -nt ${number_of_threads}
'''

# Write the shell script to disk in /content/
with open('/content/runradioss.sh', 'w') as f:
    f.write(runradioss_sh_content)

# Grant execute permission so the script can be called directly
!chmod +x /content/runradioss.sh

print("Linux run script created at /content/runradioss.sh")


Linux run script created at /content/runradioss.sh


## Step 4 — Download the Bumper Beam Input Files

Clone the Radioss input deck for the Bumper Beam model from Hugging Face. The original [OpenRadioss Bumper Beam example](http://openradioss.atlassian.net/wiki/spaces/OPENRADIOSS/pages/11075585/Bumper+Beam) outputs results in H3D format. This repository includes a **modified engine file** configured to output ANIM files instead, which are required for the ANIM-to-d3plot conversion step.


In [ ]:
# Download the Radioss input for the Bumper Beam example
!git clone https://huggingface.co/datasets/AIRBORNEPANDA/Bumper_Beam_Radioss_Input

Cloning into 'Bumper_Beam_Radioss_Input'...
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 11 (delta 0), reused 0 (delta 0), pack-reused 3 (from 1)
Receiving objects: 100% (11/11), 467.23 KiB | 21.24 MiB/s, done.


## Step 5 — Main Simulation Script

This section defines the core simulation loop. It covers three tasks:
1. **Detecting available CPU threads** to maximize solver performance.
2. **Defining helper functions** to modify OpenRadioss input files and launch simulations.
3. **Running the batch simulations** with randomized parameters.


### 5.1 — Detect Available CPU Threads

Query the system for the total number of logical CPU cores (threads). The solver will use all available threads by default. To limit CPU usage, manually set the `logical` variable to a lower value before running the simulation loop.


In [ ]:
import psutil

# Detect the number of physical CPU cores and logical threads available in this environment.
# Colab free tier typically provides 2 physical cores / 2 threads.
# The solver will use all available threads by default for maximum performance.
# To limit CPU usage, manually set `logical` to a lower integer value before running the simulation loop.
physical = psutil.cpu_count(logical=False)
logical  = psutil.cpu_count(logical=True)

print(f"Physical Cores: {physical}")
print(f"Total Threads:  {logical}")


Physical Cores: 4
Total Threads:  8


### 5.2 — Helper Functions

The following functions handle all file manipulation and simulation execution steps:

- **`format_radioss_field(value)`** — Formats a numeric value to fit the fixed-width column format required by OpenRadioss input decks.
- **`modify_radioss_starter(...)`** — Reads the master starter file (`_0000.rad`) and writes a modified copy with updated `PROP/SHELL` thicknesses and an optional rigid wall (`RWALL/CYL`) Y-position offset, enabling geometric variation across runs.
- **`copy_engine_file(...)`** — Copies the master engine file (`_0001.rad`) unchanged into each run directory, since solver settings remain constant across all runs.
- **`run_radioss(...)`** — Invokes the `runradioss.sh` script to execute the Starter and Engine sequentially for a given simulation directory.
- **`convert_anim_to_d3plot(...)`** — Converts the ANIM output files produced by the Engine into the d3plot format required by PhysicsNeMo, using the Vortex-Radioss library.


In [ ]:
import sys
import subprocess
import os
import random
from vortex_radioss.animtod3plot.Anim_to_D3plot import readAndConvert


def format_radioss_field(value):
    """
    Format a numeric value into a 10-character right-justified string
    compatible with the fixed-width column format required by OpenRadioss input decks.
    Falls back to scientific notation if the value is too large for fixed-point format.
    """
    try:
        formatted = "{:10.2f}".format(value)
        if len(formatted) > 10:
            formatted = "{:.2E}".format(value).replace('E', 'E+').replace('E+-', 'E-')
            if len(formatted) > 10:
                formatted = "{:.1E}".format(value).replace('E', 'E+').replace('E+-', 'E-')
        return formatted.rjust(10)
    except ValueError:
        return "        "


def modify_radioss_starter(input_filepath, output_filepath, prop_changes, pole_y_offset=None):
    """
    Write a modified copy of a RADIOSS starter file (_0000.rad) with updated
    PROP/SHELL thicknesses and an optional rigid wall (RWALL/CYL) Y-position offset.

    The function reads the input file line by line and rewrites it, replacing
    the thickness value on line 8 of each targeted PROP/SHELL card and the
    YM / Y_M1 values on lines 7 and 9 of the RWALL/CYL/1 card.

    Args:
        input_filepath  : Path to the master starter file to read from.
        output_filepath : Path where the modified starter file will be written.
        prop_changes    : Dict mapping PROP/SHELL ID (int) -> new thickness (float),
                          e.g. {2: 1.5, 7: 1.5}.
        pole_y_offset   : Float offset (mm) added to the YM and Y_M1 values of
                          /RWALL/CYL/1 to shift the pole impact position. Pass None
                          to leave the pole position unchanged.
    """
    try:
        if not os.path.exists(input_filepath):
            print(f"Error: Input file not found at '{input_filepath}'")
            return

        with open(input_filepath, 'r') as infile, open(output_filepath, 'w') as outfile:

            in_prop_shell = False
            in_rwall = False
            rwall_line_counter = 0
            modifications_made = {}
            prop_id_str = "9999999"
            line_counter = 0
            modified_line = ""

            for line in infile:
                # Flag to skip writing the original line when a modified version is written instead
                skip_line = False

                # Detect the start of a /PROP/SHELL block and read its ID from the card header
                if line.strip().startswith('/PROP/SHELL/'):
                    in_prop_shell = True
                    prop_id_str = line[12:].strip()

                current_prop_id = int(prop_id_str)
                if current_prop_id in prop_changes:
                    if line_counter <= 8 and in_prop_shell:
                        line_counter += 1
                        in_prop_shell = True
                    else:
                        line_counter = 0
                        in_prop_shell = False

                    # Line 8 of the PROP/SHELL card holds the thickness value (columns 31-40)
                    if line_counter == 8:
                        line_counter = 0
                        in_prop_shell = False
                        new_thickness = prop_changes[current_prop_id]
                        modified_line = line[:30] + format_radioss_field(new_thickness) + line[40:]
                        outfile.write(modified_line)
                        modifications_made[current_prop_id] = modifications_made.get(current_prop_id, 0) + 1
                        skip_line = True
                else:
                    if in_prop_shell:
                        line_counter = 0
                        in_prop_shell = False

                # Detect the start of the /RWALL/CYL/1 block
                if line.strip().startswith('/RWALL/CYL/1'):
                    in_rwall = True
                    rwall_line_counter = 0
                elif in_rwall:
                    rwall_line_counter += 1

                    # Line 7: contains the YM (Y-coordinate of the pole center) in columns 31-40
                    if rwall_line_counter == 7 and pole_y_offset is not None:
                        ym_str = line[30:40].strip()
                        try:
                            ym_val = float(ym_str)
                            new_ym = ym_val + pole_y_offset
                            modified_line = line[:30] + format_radioss_field(new_ym) + line[40:]
                            outfile.write(modified_line)
                            modifications_made['pole_YM'] = True
                            skip_line = True
                        except ValueError:
                            pass  # Leave the original line unchanged if parsing fails

                    # Line 9: contains the Y_M1 (Y-coordinate of the pole axis direction) in columns 31-40
                    elif rwall_line_counter == 9 and pole_y_offset is not None:
                        ym1_str = line[30:40].strip()
                        try:
                            ym1_val = float(ym1_str)
                            new_ym1 = ym1_val + pole_y_offset
                            modified_line = line[:30] + format_radioss_field(new_ym1) + line[40:]
                            outfile.write(modified_line)
                            modifications_made['pole_Y_M1'] = True
                            skip_line = True
                        except ValueError:
                            pass  # Leave the original line unchanged if parsing fails

                    elif rwall_line_counter >= 10:
                        # End of the RWALL block — reset state
                        in_rwall = False
                        rwall_line_counter = 0

                # Write the original line unless it was replaced by a modified version above
                if not skip_line:
                    outfile.write(line)

            # Warn if any requested PROP/SHELL IDs were not found in the file
            for prop_id in prop_changes:
                if prop_id not in modifications_made:
                    print(f"Warning: /PROP/SHELL card with ID {prop_id} was not found or updated.")

            # Warn if the pole position modifications were not applied
            if pole_y_offset is not None:
                if 'pole_YM' not in modifications_made:
                    print(f"Warning: Pole YM position was not found or updated.")
                if 'pole_Y_M1' not in modifications_made:
                    print(f"Warning: Pole Y_M1 position was not found or updated.")

            print(f"Run saved to: '{output_filepath}'")

    except FileNotFoundError:
        print(f"Error: Could not open one of the files.")
    except Exception as e:
        print(f"Oops! you have to deal with this: {e}")


def copy_engine_file(master_engine_file_with_path, new_engine_file_with_path):
    """
    Copy the master engine file (_0001.rad) unchanged into the run directory.
    Solver settings (time step, output frequency, termination time) remain
    constant across all simulation runs, so no modifications are needed.
    """
    try:
        if not os.path.exists(master_engine_file_with_path):
            print(f"Error: Input file not found at '{master_engine_file_with_path}'")
            return

        with open(master_engine_file_with_path, 'r') as infile, open(new_engine_file_with_path, 'w') as outfile:
            for line in infile:
                outfile.write(line)

    except FileNotFoundError:
        print(f"Error: Could not open one of the files.")
    except Exception as e:
        print(f"Oops! you have to deal with this: {e}")


def run_radioss(openradiosspath: str, simpath: str, simfile: str, nt: int):
    """
    Launch the OpenRadioss Starter and Engine sequentially via the generated shell script.
    Solver output is streamed directly to the notebook so progress can be monitored in real time.
    """
    command = "/content/runradioss.sh" + " " + openradiosspath + " " + simpath + " " + simfile + " " + str(nt)
    !{command}


def convert_anim_to_d3plot(resultfile_dir: str, resultfile: str):
    """
    Convert the ANIM output files produced by the OpenRadioss Engine into d3plot format
    using Vortex-Radioss. The resulting d3plot file is what PhysicsNeMo reads during training.
    """
    file_stem = resultfile_dir + "/" + resultfile
    readAndConvert(file_stem, use_shell_mask=False, silent=False)


### 5.3 — Run Simulations

Execute the full batch of simulations. For each run, the script randomly samples shell thickness and pole position values within the defined ranges, modifies the input files accordingly, runs the OpenRadioss solver, and converts the output to d3plot format.

**Configurable parameters:**

| Parameter | Description |
|---|---|
| `number_of_sims` | Total number of simulation variations to generate and run |
| `thickness_range` | Min/max shell thickness values (mm) sampled uniformly at random |
| `pole_y_range` | Min/max Y-axis offset (mm) for the rigid pole impact position |
| `prop_ids_to_change` | List of `PROP/SHELL` IDs in the input deck to update with the sampled thickness |

**Output directory structure:**
```
RAW_DATA/
├── Run100/
│   ├── Bumper_Beam_AP_meshed.d3plot
│   ├── Bumper_Beam_AP_meshed_0000.rad   # Modified starter file
│   └── Bumper_Beam_AP_meshed_0001.rad   # Engine file
├── Run101/
│   └── ...
└── ...
```


In [ ]:
print("GENERATING D3PLOT DATASET FOR PhysicsNemo")
print("       POWERED BY OpenRadioss")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION — adjust these parameters before running
# ─────────────────────────────────────────────────────────────────────────────

# Number of CPU threads to pass to the OpenRadioss solver
# Colab free tier typically provides 2 threads; Pro/Pro+ may provide more
nt = logical

# Simulation working directory — outputs are saved to local Colab storage (/content/)
sim_path = '/content'

# Path to the extracted OpenRadioss installation in the Colab environment
openradioss_path = "/content/OpenRadioss"

# Base name of the Radioss input deck (without the _0000/_0001 suffix)
sim_name = "Bumper_Beam_AP_meshed"

# Shell thickness range (mm): a single value is sampled uniformly from [min, max]
# and applied to all PROP/SHELL IDs listed in prop_ids_to_change
thickness_range = (1.2, 2.0)

# PROP/SHELL IDs to update with the sampled thickness (must match IDs in the input deck)
prop_ids_to_change = [2, 7]

# Rigid pole Y-axis offset range (mm): shifts the impact location along the beam
pole_y_range = (-500, 500)

# Total number of simulation variations to generate and execute
number_of_sims = 1

# ─────────────────────────────────────────────────────────────────────────────
# SIMULATION LOOP
# ─────────────────────────────────────────────────────────────────────────────

# Create the top-level output directory that will hold all run sub-folders
raw_data_path = os.path.join(sim_path, "RAW_DATA")
os.makedirs(raw_data_path, exist_ok=True)

# Resolve paths to the master input files (unchanged across all runs)
master_starter_file = os.path.join("/content/Bumper_Beam_Radioss_Input", sim_name + "_0000.rad")
master_engine_file  = os.path.join("/content/Bumper_Beam_Radioss_Input", sim_name + "_0001.rad")

if not os.path.exists(master_starter_file):
    print(f"ERROR: Master starter file not found: {master_starter_file}")
    sys.exit(1)

if not os.path.exists(master_engine_file):
    print(f"ERROR: Master engine file not found: {master_engine_file}")
    sys.exit(1)

print(f"Starting {number_of_sims} simulations...")
print(f"Results will be saved to: {raw_data_path}")

for i in range(number_of_sims):

    # Name run folders Run100–Run1XX so they sort correctly alphabetically
    if i < 10:
        run_dir = os.path.join(raw_data_path, f"Run10{i}")
    else:
        run_dir = os.path.join(raw_data_path, f"Run1{i}")

    os.makedirs(run_dir, exist_ok=True)
    print(f"\n{'='*60}")
    print(f"Run {i+1}/{number_of_sims}: {run_dir}")
    print(f"{'='*60}")

    # Sample a single random thickness and apply it to all targeted PROP/SHELL IDs
    new_thickness = random.uniform(thickness_range[0], thickness_range[1])
    prop_changes = {prop_id: new_thickness for prop_id in prop_ids_to_change}
    print(f"Generated thickness: {new_thickness:.4f} mm for props: {prop_ids_to_change}")

    # Sample a random Y-axis offset for the rigid pole impact position
    pole_y_offset = random.uniform(pole_y_range[0], pole_y_range[1])
    print(f"Generated pole Y offset: {pole_y_offset:.2f} mm")

    # Resolve paths for the run-specific copies of the input files
    new_starter_file = os.path.join(run_dir, sim_name + "_0000.rad")
    new_engine_file  = os.path.join(run_dir, sim_name + "_0001.rad")

    # Write the modified starter file and copy the unchanged engine file into the run folder
    modify_radioss_starter(master_starter_file, new_starter_file, prop_changes, pole_y_offset)
    copy_engine_file(master_engine_file, new_engine_file)

    # Run the OpenRadioss Starter + Engine and convert ANIM output to d3plot
    run_radioss(openradioss_path, run_dir, sim_name, nt)
    convert_anim_to_d3plot(run_dir, sim_name)

    print(f"Run {i+1} completed!")

print(f"\n{'='*60}")
print("ALL SIMULATIONS COMPLETED!")
print(f"Dataset saved to: {raw_data_path}")
print(f"{'='*60}")


GENERATING D3PLOT DATASET FOR PhysicsNemo
       POWERED BY OpenRadioss
Starting 1 simulations...
Results will be saved to: /content/RAW_DATA

Run 1/1: /content/RAW_DATA/Run100
Generated thickness: 1.5262 mm for props: [2, 7]
Generated pole Y offset: 26.56 mm
Run saved to: '/content/RAW_DATA/Run100/Bumper_Beam_AP_meshed_0000.rad'
************************************************************************
**                                                                    **
**                                                                    **
**                        OpenRadioss Starter                         **
**                                                                    **
**            Non-linear Finite Element Analysis Software             **
**                                                                    **
**                                                                    **
**                                                                    **
**         

100%|██████████| 11/11 [00:00<00:00, 29.92it/s]

Run 1 completed!

ALL SIMULATIONS COMPLETED!
Dataset saved to: /content/RAW_DATA


## Step 6 — Download the Dataset (Colab only)

Compress the entire `RAW_DATA` folder into a single zip archive and trigger a browser download. This is the recommended way to retrieve results from a Colab session before the runtime resets.

> **Note:** This step is only needed when running in Google Colab. If you are running this notebook on a local machine, the `RAW_DATA` folder is already saved in your working directory and no download is required.


In [ ]:
import shutil
from google.colab import files

# Compress the RAW_DATA folder into a zip archive in /content/
zip_path = "/content/RAW_DATA"
output_zip = "/content/RAW_DATA.zip"

print("Compressing RAW_DATA folder...")
shutil.make_archive(zip_path, 'zip', '/content', 'RAW_DATA')
print(f"Archive created at: {output_zip}")

# Trigger a browser download of the zip file
print("Starting download...")
files.download(output_zip)


Compressing RAW_DATA folder...
Archive created at: /content/RAW_DATA.zip
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 7 — Generate Global Features JSON

Parse all generated starter files (`_0000.rad`) and extract the key simulation parameters into a single `GLOBAL_FEATURES.json` file. This file acts as a lookup table mapping each run to its input conditions, which PhysicsNeMo uses as **global features** during model training.

The following parameters are extracted from each run:

| Field | Source in `_0000.rad` file | Description |
|---|---|---|
| `velocity_x` | `/INIVEL/TRA/1` | Initial impact velocity in the X direction (mm/ms) |
| `rwall_origin_y` | `/RWALL/CYL/1` | Y-coordinate of the rigid pole center (mm) |
| `thickness_scale` | `/PROP/SHELL/2` | Shell thickness applied to the bumper components (mm) |


In [ ]:
import os
import json
from pathlib import Path


def parse_rad_file(filepath):
    """
    Parse a single RADIOSS starter file (_0000.rad) and extract the three
    global features used by PhysicsNeMo during training:
      - velocity_x       : initial impact velocity in X (from /INIVEL/TRA/1)
      - rwall_origin_y   : Y-coordinate of the rigid pole center (from /RWALL/CYL/1)
      - thickness_scale  : shell thickness applied to PROP/SHELL/2 (mm)

    Returns a dict with the extracted values, or None if the file cannot be read.
    """
    data = {}
    try:
        with open(filepath, 'r') as f:
            lines = f.readlines()
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return None

    for i, line in enumerate(lines):
        line = line.strip()

        # 1. Extract initial velocity in X from the /INIVEL/TRA/1 block
        if line.startswith('/INIVEL/TRA/1'):
            for j in range(i + 1, len(lines)):
                if lines[j].startswith('/'):
                    break  # Next block started — stop searching
                if 'Vx' in lines[j] and 'Vy' in lines[j]:
                    val_line = lines[j + 1].split()
                    try:
                        data['velocity_x'] = float(val_line[0])  # Vx is the first value
                    except (IndexError, ValueError):
                        print(f"Warning: Could not parse velocity_x in {filepath}")
                    break

        # 2. Extract the rigid pole Y-coordinate from the /RWALL/CYL/1 block
        elif line.startswith('/RWALL/CYL/1'):
            for j in range(i + 1, len(lines)):
                if lines[j].startswith('/'):
                    break  # Next block started — stop searching
                if 'XM' in lines[j] and 'YM' in lines[j]:
                    val_line = lines[j + 1].split()
                    try:
                        data['rwall_origin_y'] = float(val_line[1])  # YM is the second value (index 1)
                    except (IndexError, ValueError):
                        print(f"Warning: Could not parse rwall_origin_y in {filepath}")
                    break

        # 3. Extract the shell thickness from the /PROP/SHELL/2 block
        elif line.startswith('/PROP/SHELL/2'):
            for j in range(i + 1, len(lines)):
                if lines[j].startswith('/'):
                    break  # Next block started — stop searching
                if 'Thick' in lines[j]:
                    val_line = lines[j + 1].split()
                    try:
                        data['thickness_scale'] = float(val_line[2])  # Thick is the third value (index 2)
                    except (IndexError, ValueError):
                        print(f"Warning: Could not parse thickness_scale in {filepath}")
                    break

    # Warn about any expected keys that were not found in the file
    for k in ['velocity_x', 'rwall_origin_y', 'thickness_scale']:
        if k not in data:
            print(f"Warning: Missing '{k}' in {filepath}")

    return data


def extract_simulation_data(base_dirs, output_file):
    """
    Iterate over all run sub-folders inside the given base directories,
    parse each starter file, and aggregate the results into a single JSON file.

    The output JSON maps each run folder name to its extracted parameter dict:
        { "Run100": { "velocity_x": ..., "rwall_origin_y": ..., "thickness_scale": ... }, ... }

    Args:
        base_dirs   : List of paths to scan for run sub-folders (e.g. ['./RAW_DATA']).
        output_file : Path where the resulting JSON file will be written.
    """
    results = {}

    for base_dir in base_dirs:
        base_path = Path(base_dir)
        if not base_path.exists():
            print(f"Warning: Directory '{base_dir}' does not exist. Skipping.")
            continue

        for run_dir in base_path.iterdir():
            if run_dir.is_dir():
                # Each run folder should contain exactly one *_0000.rad starter file
                rad_files = list(run_dir.glob('*0000.rad'))
                if rad_files:
                    parsed_data = parse_rad_file(rad_files[0])
                    if parsed_data:
                        results[run_dir.name] = parsed_data
                else:
                    print(f"Warning: No *0000.rad file found in '{run_dir}'")

    # Write the aggregated results to disk as a formatted JSON file
    with open(output_file, 'w') as f:
        json.dump(results, f, indent=4)
    print(f"\nExtraction complete. Data saved to '{output_file}'")


# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION AND EXECUTION
# ─────────────────────────────────────────────────────────────────────────────

# Directories to scan for run sub-folders (add more paths here if needed)
DIRECTORIES_TO_SCAN = ['./RAW_DATA']

# Output file path for the global features JSON consumed by PhysicsNeMo
OUTPUT_JSON_FILE = 'GLOBAL_FEATURES.json'

extract_simulation_data(DIRECTORIES_TO_SCAN, OUTPUT_JSON_FILE)



Extraction complete. Data saved to 'GLOBAL_FEATURES.json'
